In [1]:
import pandas as pd
import re
import requests
from datetime import date, timedelta
import unicodedata
import json
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [2]:
from unicodedata import normalize as uninorm

def normalize_string(s):
    if not isinstance(s, str):
        return s
    s = s.lower()
    s = uninorm("NFD", s)
    s = "".join(c for c in s if __import__('unicodedata').category(c) != "Mn")
    s = " ".join(s.split())
    return s


def normalizeMunicipality(term):
    _TODELETE = ['ΔΗΜΟΣ', 'Δήμος', 'Δ.', ":"]
    for i in _TODELETE:
        term = normalize_string(term).replace(normalize_string(i), "")
    replacements_raw = {
        "ΝΙΚΑΙΑΣ - ΑΓΙΟΥ Ι. ΡΕΝΤΗ": "ΝΙΚΑΙΑΣ - ΑΓΙΟΥ ΙΩΑΝΝΗ ΡΕΝΤΗ",
        "ΛΙΒΑΔΕΙΑΣ" : "ΛΕΒΑΔΕΩΝ",
        "ΚΑΡΔΑΜΥΛΩΝ": "ΧΙΟΥ",
        "ΦΙΛΛΙΠΙΑΔΑΣ": "ΖΗΡΟΥ",
        "ΑΡΤΑΣ": "ΑΡΤΑΙΩΝ",
        "ΑΙΤΩΛΙΚΟΥ": "ΙΕΡΑΣ ΠΟΛΗΣ ΜΕΣΟΛΟΓΓΙΟΥ",
        "ΠΕΙΡΑΙΑ": "ΠΕΙΡΑΙΩΣ",
        "ΑΣΤΡΟΥΣ": "ΒΟΡΕΙΑΣ ΚΥΝΟΥΡΙΑΣ",
        "ΝΕΥΡΟΚΟΠΙΟΥ": "ΚΑΤΩ ΝΕΥΡΟΚΟΠΙΟΥ",
        "ΝΑΥΠΛΙΟΥ": "ΝΑΥΠΛΙΕΩΝ",
        "ΘΗΒΑΣ": "ΘΗΒΑΙΩΝ",
        "ΛΕΧΑΙΝΩΝ": "ΑΝΔΡΑΒΙΔΑΣ - ΚΥΛΛΗΝΗΣ",
        "ΖΑΚΥΝΘΙΩΝ": "ΖΑΚΥΝΘΟΥ",
        "ΕΛΑΤΙΩΝ": "ΑΜΦΙΚΛΕΙΑΣ - ΕΛΑΤΕΙΑΣ",
        "ΙΤΕΑΣ": "ΔΕΛΦΩΝ",
        "ΑΡΕΟΠΟΛΕΩΣ": "ΑΝΑΤΟΛΙΚΗΣ ΜΑΝΗΣ",
        "ΛΑΥΡΙΟΥ": "ΛΑΥΡΕΩΤΙΚΗΣ",
        "ΔΕΛΒΙΝΑΚΙΟΥ": "ΠΩΓΩΝΙΟΥ",
        "ΚΗΡΥΚΟΥ": "ΙΚΑΡΙΑΣ",
        "ΝΑΥΠΑΚΤΟΥ":"ΝΑΥΠΑΚΤΙΑΣ",
        "ΑΡΓΟΥΣ":"ΑΡΓΟΥΣ - ΜΥΚΗΝΩΝ",
        "ΜΩΛΟΥ - ΑΓ. ΚΩΝΣΤΑΝΤΙΝΟΥ":"ΚΑΜΕΝΩΝ ΒΟΥΡΛΩΝ",
        "ΚΟΡΙΝΘΟΥ":"ΚΟΡΙΝΘΙΩΝ",
        "ΛΟΥΤΡΑΚΙΟΥ - ΑΓ. ΘΕΟΔΩΡΩΝ":"ΛΟΥΤΡΑΚΙΟΥ - ΠΕΡΑΧΩΡΑΣ - ΑΓΙΩΝ ΘΕΟΔΩΡΩΝ",
        "ΣΤΑΓΥΡΩΝ - ΑΚΑΝΘΟΥ":"ΑΡΙΣΤΟΤΕΛΗ",
        "ΔΕΣΦΙΝΗΣ":"ΔΕΛΦΩΝ",
        "ΚΑΛΛΟΝΗΣ":"ΔΥΤΙΚΗΣ ΛΕΣΒΟΥ",
        "ΓΑΡΓΑΛΙΑΝΩΝ":"ΤΡΙΦΥΛΙΑΣ",
        "ΑΘΩΣ":"ΟΡΟΥΣ",
        "ΒΑΘΕΟΣ":"ΣΑΜΟΥ",
        "ΕΥΡΩΣΤΙΝΗΣ":"ΞΥΛΟΚΑΣΤΡΟΥ - ΕΥΡΩΣΤΙΝΗΣ",
        "ΕΛΕΥΘΕΡΟΥΠΟΛΗΣ":"ΠΑΓΓΑΙΟΥ",
        "ΧΡΥΣΟΥΠΟΛΗΣ":"ΝΕΣΤΟΥ",
        "ΚΡΑΝΙΔΙΟΥ":"ΕΡΜΙΟΝΙΔΑΣ",
        "ΛΑΡΙΣΑΣ":"ΛΑΡΙΣΑΙΩΝ",
        "ΕΠΤΑΧΩΡΙΟΥ":"ΒΟΙΟΥ",
        "ΒΡΥΣΣΩΝ":"ΑΠΟΚΟΡΩΝΟΥ",
        "ΔΙΚΑΙΩΝ":"ΟΡΕΣΤΙΑΔΑΣ",
        "ΘΕΣΠΡΩΤΙΚΟΥ":"ΖΗΡΟΥ",
        "ΜΕΤΑΜΟΡΦΩΣΕΩΣ": "ΜΕΤΑΜΟΡΦΩΣΗΣ",
        "ΑΚΤΙΟΥ ΒΟΝΙΤΣΑΣ": "ΑΚΤΙΟΥ - ΒΟΝΙΤΣΑΣ",
        "ΑΡΓΟΣ ΟΡΕΣΤΙΚΟΥ": "ΟΡΕΣΤΙΔΟΣ",
        "ΠΕΤΡΟΥΠΟΛΕΩΣ": "ΠΕΤΡΟΥΠΟΛΗΣ",
        "ΛΕΥΚΑΔΟΣ": "ΛΕΥΚΑΔΑΣ",
        "ΛΟΥΤΡΑΚΙΟΥ": "ΔΗΜΟΣ ΛΟΥΤΡΑΚΙΟΥ - ΠΕΡΑΧΩΡΑΣ - ΑΓΙΩΝ ΘΕΟΔΩΡΩΝ"
    }
    replacements = {normalize_string(k): v for k, v in replacements_raw.items()}
    
    term = re.sub(r"\s*-\s*", " - ", term)
    term = term.replace("(", "").replace(")", "")
    term = replacements.get(term, term)
    return term.upper().strip()

In [3]:
df = pd.read_csv("../data/raw_procurements.csv")
diavgeia = pd.read_csv("../data/2026_diavgeia.csv")
fires = pd.read_csv("../data/fires/fire_incidents_unified.csv")
funding = pd.read_csv("../data/funding/municipal_funding.csv")
fires['municipality_name_canonical'] = fires['municipality_name_canonical'].fillna(fires['municipality_raw'])
fires['municipality_name_canonical'] = fires['municipality_name_canonical'].apply(normalizeMunicipality)
funding['municipality_code'] = funding['municipality_code'].astype("Int64")

In [4]:
funding[1000:1200]

,year,allocation_type,recipient_type,recipient_raw,nomos,municipality_code,amount_eur,source_file,source_ada
1000,2019,τακτική,δήμος,ΤΡΙΚΚΑΙΩΝ,ΤΡΙΚΑΛΩΝ,54425,57000.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1001,2019,τακτική,δήμος,ΦΑΡΚΑΔΟΝΑΣ,ΤΡΙΚΑΛΩΝ,54421,21000.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1002,2019,τακτική,δήμος,ΑΜΦΙΚΛΕΙΑΣ-ΕΛΑΤΕΙΑΣ,ΦΘΙΩΤΙΔΟΣ,50721,66000.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1003,2019,τακτική,δήμος,ΔΟΜΟΚΟΥ,ΦΘΙΩΤΙΔΟΣ,50706,49000.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1004,2019,τακτική,δήμος,ΛΑΜΙΕΩΝ,ΦΘΙΩΤΙΔΟΣ,50711,109000.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1005,2019,τακτική,δήμος,ΛΟΚΡΩΝ,ΦΘΙΩΤΙΔΟΣ,50703,83700.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1006,2019,τακτική,δήμος,ΜΑΚΡΑΚΩΜΗΣ,ΦΘΙΩΤΙΔΟΣ,50719,75000.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1007,2019,τακτική,δήμος,ΚΑΜΕΝΩΝ ΒΟΥΡΛΩΝ,ΦΘΙΩΤΙΔΟΣ,50710,52000.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1008,2019,τακτική,δήμος,ΣΤΥΛΙΔΑΣ,ΦΘΙΩΤΙΔΟΣ,50720,47600.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368
1009,2019,τακτική,δήμος,ΑΜΥΝΤΑΙΟΥ,ΦΛΩΡΙΝΗΣ,56302,45000.0,6ΔΖΟ465ΧΘ7-368_τακτική_δήμοι_2019.pdf,6ΔΖΟ465ΧΘ7-368


In [5]:
procurement_municipalities = list(set(df[df.organization_value.str.contains("ΔΗΜΟΣ ")]["organization_value"].tolist()))
len(procurement_municipalities)

291

In [6]:
diavgeia_municipalities = list(set(diavgeia[diavgeia.org.str.contains("ΔΗΜΟΣ ")]["org"].tolist()))
len(diavgeia_municipalities)

279

In [7]:
fires_municipalities = list(set(fires.municipality_name_canonical.tolist()))
len(fires_municipalities)

319

In [8]:
funding_municipalities = []
for i, item in funding.iterrows():
    if item['recipient_type'] == "δήμος":
        funding_municipalities.append(item['recipient_raw'])
funding_municipalities = list(set(funding_municipalities))
len(funding_municipalities)

412

In [447]:
funding_municipalities[:5]

['ΛΕΣΒΟΥ', 'ΠΑΡΓΑΣ', 'ΜΕΣΣΗΝΗΣ', 'ΣΑΜΗΣ', 'ΖΑΚΥΝΘΟΥ']

In [9]:
old_mp = pd.read_csv("/Users/troboukis/Downloads/final_entity_mapping_expanded.csv")
mp = pd.read_csv("../data/mappings/final_entity_mapping_expanded.csv")
mp["municipality_id"] = mp["municipality_id"].astype("Int64")
# mp['normalized_value'] = mp.normalized_value.apply(normalizeMunicipality)
mp.head(1)

,source_system,source_entity_type,source_key,source_value,normalized_value,municipality_id,municipality_name,region_id,match_strategy,notes
0,geo,municipality,9001,Δοξάτου,ΔΟΞΑΤΟΥ,9001,Δοξάτου,ΑΝΑΤΟΛΙΚΗΣ ΜΑΚΕΔΟΝΙΑΣ ΚΑΙ ΘΡΑΚΗΣ,geo_source_of_truth,NaN


In [14]:
mp.groupby("source_system").source_entity_type.value_counts()

source_system  source_entity_type
diavgeia       organization           8253
               municipality            885
fires          municipality            974
               municipality_name       721
funding        municipality           2043
               municipality_name       417
               organization             25
geo            municipality            326
procurement    municipality            889
procurements   organization          14978
Name: count, dtype: int64

In [449]:
# r = []
# for i in procurement_municipalities:
#     j={}
#     j['source_system'] = 'procurement'
#     j['source_entity_type'] = 'municipality'
#     j['source_value'] = i
#     j['normalized_value'] = normalizeMunicipality(i)
#     r.append(j)

# k = pd.DataFrame(r)
# k.head(2)

In [450]:
# r = []
# for i in diavgeia_municipalities:
#     j={}
#     j['source_system'] = 'diavgeia'
#     j['source_entity_type'] = 'municipality'
#     j['source_value'] = i
#     j['normalized_value'] = normalizeMunicipality(i)
#     r.append(j)

# k = pd.DataFrame(r)
# k.head(2)

In [451]:
# r = []
# for i in fires_municipalities:
#     j={}
#     j['source_system'] = 'fires'
#     j['source_entity_type'] = 'municipality'
#     j['source_value'] = i
#     j['normalized_value'] = normalizeMunicipality(i)
#     r.append(j)

# k = pd.DataFrame(r)
# k.head(2)

In [452]:
# r = []
# for i in funding_municipalities:
#     j={}
#     j['source_system'] = 'funding'
#     j['source_entity_type'] = 'municipality'
#     j['source_value'] = i
#     j['normalized_value'] = normalizeMunicipality(i)
#     r.append(j)

# k = pd.DataFrame(r)
# k.head(2)

,source_system,source_entity_type,source_value,normalized_value
0,funding,municipality,ΛΕΣΒΟΥ,ΛΕΣΒΟΥ
1,funding,municipality,ΠΑΡΓΑΣ,ΠΑΡΓΑΣ


In [453]:
l = pd.merge(k, mp, how='left', left_on='normalized_value', right_on='normalized_value',suffixes=('', '_mp'))
l = l.drop(columns=["source_system_mp", "source_entity_type_mp", "source_value_mp"])

In [454]:
l.head(1)

,source_system,source_entity_type,source_value,normalized_value,source_key,municipality_id,municipality_name,region_id,match_strategy,notes
0,funding,municipality,ΛΕΣΒΟΥ,ΛΕΣΒΟΥ,9261,9261,Λέσβου,ΒΟΡΕΙΟΥ ΑΙΓΑΙΟΥ,geo_source_of_truth,NaN


In [455]:
new_mp = pd.concat([mp, l], ignore_index=True)

In [459]:
new_mp.shape

(29511, 10)

In [457]:
new_mp['normalized_value'] = new_mp['source_value'].apply(normalizeMunicipality)

In [458]:
new_mp = new_mp.drop_duplicates().reset_index(drop=True)

In [460]:
new_mp.to_csv("../data/mappings/final_entity_mapping_expanded.csv", index=False)

In [434]:
new_mp.head(1)

,source_system,source_entity_type,source_key,source_value,normalized_value,municipality_id,municipality_name,region_id,match_strategy,notes
0,geo,municipality,9001,Δοξάτου,ΔΟΞΑΤΟΥ,9001,Δοξάτου,ΑΝΑΤΟΛΙΚΗΣ ΜΑΚΕΔΟΝΙΑΣ ΚΑΙ ΘΡΑΚΗΣ,geo_source_of_truth,NaN


In [435]:
new_mp.source_system.value_counts()

source_system
procurements    14978
diavgeia         9138
funding          2465
fires            1695
procurement       889
geo               326
Name: count, dtype: int64

In [436]:
f = list(set(new_mp[(new_mp['source_system']=='funding') & (new_mp['normalized_value'].notna())]['normalized_value'].tolist()))
p = list(set(new_mp[(new_mp['source_system']=='procurement') & (new_mp['normalized_value'].notna())]['normalized_value'].tolist()))
print(f"funding has {len(f)} municipalities\nprocurement has {len(p)} municipalities")

funding has 362 municipalities
procurement has 291 municipalities


In [437]:
df.head(1)

,title,referenceNumber,submissionDate,contractSignedDate,startDate,noEndDate,endDate,cancelled,cancellationDate,cancellationType,cancellationReason,decisionRelatedAda,contractNumber,organizationVatNumber,greekOrganizationVatNumber,diavgeiaADA,budget,contractBudget,bidsSubmitted,maxBidsSubmitted,numberOfSections,centralGovernmentAuthority,nutsCode_key,nutsCode_value,organization_key,organization_value,procedureType_key,procedureType_value,awardProcedure,nutsCity,nutsPostalCode,centralizedMarkets,contractType,assignCriteria,classificationOfPublicLawOrganization,typeOfContractingAuthority,contractingAuthorityActivity,contractDuration,contractDurationUnitOfMeasure,contractRelatedADA,fundingDetails_cofund,fundingDetails_selfFund,fundingDetails_espa,fundingDetails_regularBudget,unitsOperator,signers,firstMember_vatNumber,firstMember_name,totalCostWithVAT,totalCostWithoutVAT,shortDescriptions,cpv_keys,cpv_values,greenContracts,auctionRefNo,paymentRefNo
0,ΜΙΣΘΩΣΗ ΙΔΙΩΤΙΚΩΝ ΜΗΧΑΝΗΜΑΤΩΝ ΓΙΑ ΕΚΤΕΛΕΣΗ ΕΡΓΑΣΙΩΝ ΕΚΤΑΚΤΩΝ ΑΝΑΓΚΩΝ(ΒΙΕΝΝΑ ΧΡΙΣΤΙΝΑ),24SYMV014279531,2024-02-17T12:45:08.845,0023-09-17,0023-09-17,False,NaN,False,NaN,NaN,NaN,NaN,ΧΩΡΙΣ ΠΡΩΤΟΚΟΛΛΟ,997947718,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EL641,Βοιωτία,5013,ΠΕΡΙΦΕΡΕΙΑ ΣΤΕΡΕΑΣ ΕΛΛΑΔΑΣ,6.0,Απευθείας ανάθεση (αρ.118/αρ. 328),NaN,NaN,NaN,NaN,Υπηρεσίες,NaN,NaN,NaN,NaN,NaN,NaN,Ψ2Κ37ΛΗ-ΡΚΜ,NaN,NaN,NaN,NaN,ΤΜΗΜΑ ΠΟΛΙΤΙΚΗΣ ΠΡΟΣΤΑΣΙΑΣ ΠΕ ΒΟΙΩΤΙΑΣ,ΦΑΝΗΣ ΣΠΑΝΟΣ - ΠΕΡΙΦΕΡΕΙΑΡΧΗΣ,159003964,ΒΙΕΝΝΑ ΧΡΙΣΤΙΝΑ,9721.6,7840.0,ΕΡΓΑΣΙΕΣ ΠΡΟΛΗΨΗΣ ΔΑΣΙΚΩΝ ΠΥΡΚΑΓΙΩΝ ΣΤΗΝ ΕΥΡΥΤΕΡΗ ΠΕΡΙΟΧΗ ΤΗΣ Π.Ε ΒΟΙΩΤΙΑΣ,75251110-4,Υπηρεσίες πρόληψης πυρκαγιών,Δεν εμπίπτει στο ΕΣΔ,NaN,[]


In [438]:
[x for x in f if x not in p]

['ΛΕΣΒΟΥ',
 'ΑΣΤΥΠΑΛΑΙΑΣ',
 'ΝΙΣΥΡΟΥ',
 'ΚΑΛΥΜΝΙΩΝ',
 'ΦΙΛΑΔΕΛΦΕΙΑΣ - ΧΑΛΚΗΔΟΝΟΣ',
 'ΜΙΝΩΑ ΠΕΔΙΑΔΑΣ',
 'ΑΝΑΠΤΥΞΙΑΚΟΣ ΣΥΝΔΕΣΜΟΣ ΔΥΤΙΚΗΣ ΑΘΗΝΑΣ ΑΣΔΑ',
 'ΑΓΑΘΟΝΗΣΙΟΥ',
 'ΧΑΙΔΑΡΙΟΥ',
 'ΣΟΦΑΔΩΝ',
 'ΜΕΤΣΟΒΟΥ',
 'ΣΙΝΤΙΚΗΣ',
 'ΑΓΙΟΥ ΝΙΚΟΛΑΟΥ',
 'ΣΥΝΔΕΣΜΟΣ ΔΗΜΩΝ & ΚΟΙΝΟΤΗΤΩΝ ΠΡΟΣΤΑΣΙΑΣ ΚΑΙ ΑΝΑΠΛΑΣΗΣ ΤΟΥ ΠΕΡΙΒΑΛΛΟΝΤΟΣ ΤΗΣ ΠΕΡΙΟΧΗΣ ΤΟΥ ΠΕΝΤΕΛΙΚΟΥ ΑΤΤΙΚΗΣ ΣΠΑΠ',
 'ΗΛΙΟΥΠΟΛΕΩΣ',
 'ΜΑΝΔΡΑΣ - ΕΙΔΥΛΛΙΑΣ',
 'ΜΑΡΩΝΕΙΑΣ - ΣΑΠΩΝ',
 'ΚΙΛΕΛΕΡ',
 'ΧΑΛΚΗΔΟΝΟΣ',
 'ΠΥΔΝΑΣ - ΚΟΛΙΝΔΡΟΥ',
 'ΑΝΑΦΗΣ',
 'ΜΕΓΙΣΤΗΣ',
 'ΑΚΤΙΟΥ - ΒΟΝΙΤΣΑΣ',
 'ΑΚΤΙΟΥ ΒΟΝΙΤΣΑΣ',
 'ΔΕΣΚΑΤΗΣ',
 'ΦΟΛΕΓΑΝΔΡΟΥ',
 'ΑΡΓΟΥΣ ΟΡΕΣΤΙΚΟΥ',
 'ΚΕΦΑΛΟΝΙΑΣ',
 'ΑΡΓΟΣ ΟΡΕΣΤΙΚΟΥ',
 'ΝΑΥΠΑΚΤΙΑΣ',
 'ΠΑΤΜΟΥ',
 'ΞΗΡΟΜΕΡΟΥ',
 'ΑΝΑΠΤΥΞΙΑΚΟΣ ΣΥΝΔΕΣΜΟΣ ΛΑΥΡΕΩΤΙΚΗΣ',
 'ΑΛΕΞΑΝΔΡΕΙΑΣ',
 'ΠΕΡΙΒΑΛΛΟΝΤΙΚΟΣ ΣΥΝΔΕΣΜΟΣ ΔΗΜΩΝ ΑΘΗΝΑΣ - ΠΕΙΡΑΙΑ ΠΕΣΥΔΑΠ',
 'ΑΙΓΙΝΑΣ',
 'ΚΕΝΤΡΙΚΗΣ ΚΕΡΚΥΡΑΣ & ΔΙΑΠΟΝΤΙΩΝ ΝΗΣΩΝ',
 'ΠΕΤΡΟΥΠΟΛΕΩΣ',
 'ΚΑΛΑΜΠΑΚΑΣ',
 'ΛΕΥΚΑΔΟΣ',
 'ΣΗΤΕΙΑΣ',
 'ΝΑΟΥΣΑΣ',
 'ΣΑΜΟΥ',
 'ΕΜΜΑΝΟΥΗΛ ΠΑΠΠΑ',
 'ΘΗΡΑΣ',
 'ΤΡΙΦΥΛΙΑΣ',
 'ΚΕΦΑΛΛΟΝΙΑΣ',
 'ΣΥΝΔΕΣΜΟΣ ΟΤΑ ΔΗΜΩΝ ΒΑΘΕΩΣ ΠΥΘΑΓ

# Analysis validation

In [265]:
df['contractSignedDate'] = pd.to_datetime(df['contractSignedDate'])
df = df.sort_values(by = 'contractSignedDate', ascending=True)
latest_date = df.tail(1)['contractSignedDate'].item()
one_year_before = latest_date - pd.DateOffset(years=1)
two_year_before = latest_date - pd.DateOffset(years=2)
print(f"latest date in the data: {latest_date}. \nOne year before = {one_year_before}. \nTwo years before = {two_year_before}")
df.tail(1)

latest date in the data: 2026-03-04 00:00:00. 
One year before = 2025-03-04 00:00:00. 
Two years before = 2024-03-04 00:00:00


,title,referenceNumber,submissionDate,contractSignedDate,startDate,noEndDate,endDate,cancelled,cancellationDate,cancellationType,cancellationReason,decisionRelatedAda,contractNumber,organizationVatNumber,greekOrganizationVatNumber,diavgeiaADA,budget,contractBudget,bidsSubmitted,maxBidsSubmitted,numberOfSections,centralGovernmentAuthority,nutsCode_key,nutsCode_value,organization_key,organization_value,procedureType_key,procedureType_value,awardProcedure,nutsCity,nutsPostalCode,centralizedMarkets,contractType,assignCriteria,classificationOfPublicLawOrganization,typeOfContractingAuthority,contractingAuthorityActivity,contractDuration,contractDurationUnitOfMeasure,contractRelatedADA,fundingDetails_cofund,fundingDetails_selfFund,fundingDetails_espa,fundingDetails_regularBudget,unitsOperator,signers,firstMember_vatNumber,firstMember_name,totalCostWithVAT,totalCostWithoutVAT,shortDescriptions,cpv_keys,cpv_values,greenContracts,auctionRefNo,paymentRefNo
4754,"Σύμβαση αξίας 30.690,00 με φπα 24% για Υποστηρικτικές εργασίες για τη σύνταξη μελέτης για τον σχεδι",26SYMV018579344,2026-03-04T14:04:30.487,2026-03-04,2026-03-04,False,2026-06-03,False,NaN,NaN,NaN,NaN,76715,090273987,True,NaN,NaN,24750.0,NaN,2.0,1.0,Ναι,EL611,"Καρδίτσα, Τρίκαλα",100015996,ΥΠΟΥΡΓΕΙΟ ΠΕΡΙΒΑΛΛΟΝΤΟΣ ΚΑΙ ΕΝΕΡΓΕΙΑΣ,6.0,Απευθείας ανάθεση,NaN,ΚΑΛΑΜΠΑΚΑ,42200,Δεν χρησιμοποιείται,Υπηρεσίες,Βάσει τιμής,Κεντρική Κυβέρνηση,Κεντρική Διοίκηση,Γενικές δημόσιες υπηρεσίες,3.0,Μήνες,ΨΑΧ646Ψ844-ΩΕΡ,NaN,"02.001.2310189, 02.001.23104",NaN,NaN,ΔΑΣΑΡΧΕΙΟ ΚΑΛΑΜΠΑΚΑΣ,ΖΩΗ ΦΤΙΚΑ - Αναπληρωτής Γενικός Διευθυντής,042770790,ΜΟΥΛΑΣ ΓΕΩΡΓΙΟΣ,30690.0,24750.0,"Σύμβαση με τίτλο: «Υποστηρικτικές εργασίες για τη σύνταξη μελέτης για τον σχεδιασμό αντιπυρικής προστασίας στα περιαστικά Δάση Προφήτη Ηλία Τ.Κ. Καλαμπάκας και παρυφών των βράχων των Μετεώρων», αρμοδιότητας του Δασαρχείου Καλαμπάκας, στο πλαίσιο του Χρηματοδοτικού Προγράμματος του Πράσινου Ταμείου “Προστασία και αναβάθμιση Δασών 2026” Άξονας Προτεραιότητας 5, Μέτρο , Έργο 53",75251110-4,Υπηρεσίες πρόληψης πυρκαγιών,Δεν εμπίπτει στο ΕΣΔ,26AWRD018562637,[]


In [266]:
# missing dates - false = δεν λείπουν ημερομηνίες
df.contractSignedDate.isnull().value_counts()

contractSignedDate
False    4755
Name: count, dtype: int64

In [267]:
# hero section data validation

In [268]:
df_2026 = df[df['contractSignedDate'].between(pd.to_datetime("2026-01-01"), latest_date)]
df_2025 = df[df['contractSignedDate'].between(pd.to_datetime("2025-01-01"), one_year_before)]
df_2024 = df[df['contractSignedDate'].between(pd.to_datetime("2024-01-01"), two_year_before)]
print(f"{df_2026.shape[0]} εγγραφές από 1/1/2026 έως {latest_date}\n{df_2025.shape[0]} εγγραφές από 1/1/2025 έως {one_year_before}\n{df_2024.shape[0]} εγγραφές από 1/1/2024 έως {two_year_before}")

65 εγγραφές από 1/1/2026 έως 2026-03-04 00:00:00
117 εγγραφές από 1/1/2025 έως 2025-03-04 00:00:00
133 εγγραφές από 1/1/2024 έως 2024-03-04 00:00:00


In [269]:
spent_2026 = df_2026.totalCostWithoutVAT.sum().item()
spent_2025 = df_2025.totalCostWithoutVAT.sum().item()
spent_2024 = df_2024.totalCostWithoutVAT.sum().item()
change_cmp_2025 = round(((spent_2026 - spent_2025)/spent_2025)*100,1)
change_cmp_2024 = round(((spent_2026 - spent_2024)/spent_2024)*100,1)
print(f"Up until {latest_date} were spent {df_2026.totalCostWithoutVAT.sum().item()} euros\nThe same timeframe in 2025 were spent {df_2025.totalCostWithoutVAT.sum().item()}\nThe same timeframe in 2024 were spent {df_2024.totalCostWithoutVAT.sum().item()}")
print(f"In 2026 have been spent {change_cmp_2025}% compared to 2025 and {change_cmp_2024}% compared to 2024")

Up until 2026-03-04 00:00:00 were spent 10319597.62 euros
The same timeframe in 2025 were spent 75304902.21000001
The same timeframe in 2024 were spent 78837782.85
In 2026 have been spent -86.3% compared to 2025 and -86.9% compared to 2024


In [270]:
# Συχνότερη διαδικασία ανάθεσης
vc_2026 = df_2026.procedureType_value.value_counts()
top_type_2026 = vc_2026.idxmax()
top_count_2026 = vc_2026.max()

vc_2025 = df_2025.procedureType_value.value_counts()
top_type_2025 = vc_2025.idxmax()
top_count_2025 = vc_2025.max()

vc_2024 = df_2024.procedureType_value.value_counts()
top_type_2024 = vc_2024.idxmax()
top_count_2024 = vc_2024.max()

print(f"In 2026 the most common procedure type is {top_type_2026} with {top_count_2026} contracts\nIn 2025 the most common procedure type is {top_type_2025} with {top_count_2025} contracts\nIn 2024 the most common procedure type is {top_type_2024} with {top_count_2024} contracts")

In 2026 the most common procedure type is Απευθείας ανάθεση with 32 contracts
In 2025 the most common procedure type is Απευθείας ανάθεση (αρ.118/αρ. 328) with 58 contracts
In 2024 the most common procedure type is Απευθείας ανάθεση (αρ.118/αρ. 328) with 78 contracts


In [271]:
# ContractsPage
today = pd.Timestamp.today()
table_data = df[df['contractSignedDate'].between(latest_date-pd.DateOffset(days=27), today)].sort_values(by='contractSignedDate', ascending=False)[["contractSignedDate","organization_value", "title", "cpv_values", "firstMember_name", "procedureType_value", "totalCostWithoutVAT"]]
print(f"table data has {table_data.shape[0]} rows. Spanning from {latest_date-pd.DateOffset(days=27)} to {today}")
table_data.head(1)

table data has 38 rows. Spanning from 2026-02-05 00:00:00 to 2026-03-06 11:44:50.227413


,contractSignedDate,organization_value,title,cpv_values,firstMember_name,procedureType_value,totalCostWithoutVAT
4754,2026-03-04,ΥΠΟΥΡΓΕΙΟ ΠΕΡΙΒΑΛΛΟΝΤΟΣ ΚΑΙ ΕΝΕΡΓΕΙΑΣ,"Σύμβαση αξίας 30.690,00 με φπα 24% για Υποστηρικτικές εργασίες για τη σύνταξη μελέτης για τον σχεδι",Υπηρεσίες πρόληψης πυρκαγιών,ΜΟΥΛΑΣ ΓΕΩΡΓΙΟΣ,Απευθείας ανάθεση,24750.0


In [272]:
df_2026[df_2026.procedureType_value=="Απευθείας ανάθεση (αρ.118/αρ. 328)"]

,title,referenceNumber,submissionDate,contractSignedDate,startDate,noEndDate,endDate,cancelled,cancellationDate,cancellationType,cancellationReason,decisionRelatedAda,contractNumber,organizationVatNumber,greekOrganizationVatNumber,diavgeiaADA,budget,contractBudget,bidsSubmitted,maxBidsSubmitted,numberOfSections,centralGovernmentAuthority,nutsCode_key,nutsCode_value,organization_key,organization_value,procedureType_key,procedureType_value,awardProcedure,nutsCity,nutsPostalCode,centralizedMarkets,contractType,assignCriteria,classificationOfPublicLawOrganization,typeOfContractingAuthority,contractingAuthorityActivity,contractDuration,contractDurationUnitOfMeasure,contractRelatedADA,fundingDetails_cofund,fundingDetails_selfFund,fundingDetails_espa,fundingDetails_regularBudget,unitsOperator,signers,firstMember_vatNumber,firstMember_name,totalCostWithVAT,totalCostWithoutVAT,shortDescriptions,cpv_keys,cpv_values,greenContracts,auctionRefNo,paymentRefNo


In [109]:
table_data.duplicated().sum()

np.int64(0)

In [110]:
df.duplicated().sum()

np.int64(0)